# A weight-encoded scientific prior and sequential revision

This is the second reviewer-facing demonstration.

The first notebook showed that merely stating an incompatible law in context did not
overcome decisive numerical evidence. Here the prior is instead introduced through an
80-step LoRA adapter.

The experiment asks:

> Can a model retrieve a learned scientific law before seeing new evidence, preserve it
> while observations remain compatible, and revise it when controlled contradictions
> accumulate?

The taught law is

$$
L_A:\qquad \tau=3\frac{mr^2}{q},
$$

while the new observations follow

$$
L_B:\qquad \tau=3\frac{mr}{q^2}.
$$

The default cached mode verifies and analyzes the checked-in training data, adapter,
evaluations, and all six sequential runs. It requires no model download.

After cloning the repository:

```bash
python -m pip install -e ".[notebook]"
jupyter lab notebooks/02_weight_prior_and_sequential_revision.ipynb
```

In [1]:
from pathlib import Path
import hashlib
import json
import re
import sys

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (
            candidate / "src" / "prior_evidence"
        ).exists():
            return candidate
    raise RuntimeError("Could not locate the repository root.")


ROOT = find_repo_root(Path.cwd())
SRC = ROOT / "src"
EXPERIMENT = ROOT / "experiments" / "02_weight_prior"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Reviewer controls.
MODE = "cached"  # "cached" or "live_mlx"
DATASET_SEED = 7100  # 7100, 7200, or 7300
ARM = "weights"  # "base" or "weights"
STAGE = 3  # 0=preregistration; 1..4=evidence batches
SHOW_FULL_REASONING = False

if MODE not in {"cached", "live_mlx"}:
    raise ValueError("MODE must be 'cached' or 'live_mlx'")
if DATASET_SEED not in {7100, 7200, 7300}:
    raise ValueError("Cached seeds are 7100, 7200, and 7300")
if ARM not in {"base", "weights"}:
    raise ValueError("ARM must be 'base' or 'weights'")
if STAGE not in {0, 1, 2, 3, 4}:
    raise ValueError("STAGE must be an integer from 0 through 4")

print(f"Repository: {ROOT.name}")
print(
    f"Mode={MODE}, seed={DATASET_SEED}, arm={ARM}, "
    f"stage={STAGE}, full_reasoning={SHOW_FULL_REASONING}"
)

Repository: prior-evidence-diagnostic
Mode=cached, seed=7100, arm=weights, stage=3, full_reasoning=False


## 1. Verify the experiment block

The adapter, materialized curriculum, training log, summaries, and raw runs are all
checked in. Verify every file against the experiment's SHA-256 manifest before using
the results.

In [2]:
def verify_checksums(experiment_dir: Path) -> list[str]:
    verified = []
    for line in (experiment_dir / "SHA256SUMS").read_text().splitlines():
        expected, relative_path = line.split(maxsplit=1)
        payload = (experiment_dir / relative_path).read_bytes()
        observed = hashlib.sha256(payload).hexdigest()
        if observed != expected:
            raise ValueError(f"Checksum mismatch: {relative_path}")
        verified.append(relative_path)
    return verified


verified = verify_checksums(EXPERIMENT)
adapter_size_mb = (
    EXPERIMENT / "adapter" / "adapters.safetensors"
).stat().st_size / 1024**2
print(f"Verified {len(verified)} files, including the {adapter_size_mb:.1f} MB adapter.")

Verified 23 files, including the 37.0 MB adapter.


## 2. What was trained?

Only the selected diversified 80-step adapter is reported. Intermediate checkpoints and
abandoned training variants are not part of this experiment block.

The student prompts do not state $L_A$. Target completions teach the law across several
surface forms and applications. Retention examples use disjoint fictional domains and
ask the model to infer other multiplicative laws from controlled evidence.

In [3]:
adapter_config = json.loads(
    (EXPERIMENT / "adapter" / "adapter_config.json").read_text()
)
curriculum_manifest = json.loads(
    (EXPERIMENT / "training" / "data" / "manifest.json").read_text()
)
adapter_summary = json.loads(
    (EXPERIMENT / "results" / "adapter_summary.json").read_text()
)

split_sizes = curriculum_manifest["validation"]["split_sizes"]
kind_counts = curriculum_manifest["validation"]["kind_counts"]

config_table = [
    "| property | value |",
    "|---|---:|",
    f"| Base model | `{adapter_config['model']}` |",
    f"| Iterations | {adapter_config['iters']} |",
    f"| LoRA layers | {adapter_config['num_layers']} |",
    f"| Rank | {adapter_config['lora_parameters']['rank']} |",
    f"| Scale | {adapter_config['lora_parameters']['scale']} |",
    f"| Learning rate | {adapter_config['learning_rate']:.1e} |",
    f"| Effective batch size | "
    f"{adapter_config['batch_size'] * adapter_config['grad_accumulation_steps']} |",
    f"| Maximum sequence length | {adapter_config['max_seq_length']} |",
    f"| Prompt masked from loss | {adapter_config['mask_prompt']} |",
    f"| Peak training memory | "
    f"{adapter_summary['training']['peak_memory_gb']:.3f} GB |",
]
display(Markdown("\n".join(config_table)))

split_table = [
    "| split | total | target-law | retention |",
    "|---|---:|---:|---:|",
]
for split in ("train", "valid", "test"):
    split_table.append(
        f"| {split} | {split_sizes[split]} | "
        f"{kind_counts[split]['target']} | {kind_counts[split]['retain']} |"
    )
display(Markdown("\n".join(split_table)))

| property | value |
|---|---:|
| Base model | `mlx-community/Qwen3-8B-4bit` |
| Iterations | 80 |
| LoRA layers | 16 |
| Rank | 8 |
| Scale | 16.0 |
| Learning rate | 2.0e-05 |
| Effective batch size | 4 |
| Maximum sequence length | 2048 |
| Prompt masked from loss | True |
| Peak training memory | 8.051 GB |

| split | total | target-law | retention |
|---|---:|---:|---:|
| train | 28 | 24 | 4 |
| valid | 7 | 6 | 1 |
| test | 7 | 6 | 1 |

### Training trajectory

The table below is parsed from the original MLX training log rather than copied from a
handwritten summary.

In [4]:
training_log = (EXPERIMENT / "artifacts" / "training.log").read_text()
validation_points = {
    int(step): float(loss)
    for step, loss in re.findall(
        r"Iter (\d+): Val loss ([0-9.]+)",
        training_log,
    )
}
training_points = {
    int(step): {
        "loss": float(loss),
        "peak_memory_gb": float(memory),
    }
    for step, loss, memory in re.findall(
        r"Iter (\d+): Train loss ([0-9.]+).*?Peak mem ([0-9.]+) GB",
        training_log,
    )
}
test_match = re.search(r"Test loss ([0-9.]+), Test ppl ([0-9.]+)", training_log)
if test_match is None:
    raise ValueError("Could not parse final test metrics")

loss_table = [
    "| iteration | train loss | validation loss | peak memory (GB) |",
    "|---:|---:|---:|---:|",
]
for step in (20, 40, 60, 80):
    loss_table.append(
        f"| {step} | {training_points[step]['loss']:.3f} | "
        f"{validation_points[step]:.3f} | "
        f"{training_points[step]['peak_memory_gb']:.3f} |"
    )
loss_table.append(
    f"| final test | {float(test_match.group(1)):.3f} | — | — |"
)
display(Markdown("\n".join(loss_table)))

| iteration | train loss | validation loss | peak memory (GB) |
|---:|---:|---:|---:|
| 20 | 0.468 | 0.433 | 7.974 |
| 40 | 0.356 | 0.385 | 7.974 |
| 60 | 0.200 | 0.338 | 8.051 |
| 80 | 0.284 | 0.326 | 8.051 |
| final test | 0.328 | — | — |

### Inspect the actual curriculum

The examples below come directly from the materialized training JSONL. The teacher
reasoning is stored separately from the final answer, and prompt tokens are masked from
the supervised loss.

In [5]:
def load_jsonl(path: Path) -> list[dict]:
    return [
        json.loads(line)
        for line in path.read_text().splitlines()
        if line.strip()
    ]


train_records = load_jsonl(
    EXPERIMENT / "training" / "data" / "train.jsonl"
)
target_example = next(
    row
    for row in train_records
    if row["metadata"]["distillation_kind"] == "target"
    and row["metadata"]["verification_type"] == "equation"
)
retain_example = next(
    row
    for row in train_records
    if row["metadata"]["distillation_kind"] == "retain"
)


def render_training_example(record: dict, label: str) -> None:
    messages = record["messages"]
    assistant = messages[-1]
    reasoning = assistant.get("reasoning_content", "")
    body = (
        f"### {label}\n\n"
        f"**System**\n\n{messages[0]['content']}\n\n"
        f"**User**\n\n{messages[1]['content']}\n\n"
        f"**Assistant completion**\n\n```text\n"
        f"{assistant['content']}\n```\n\n"
        f"<details><summary>Stored teacher reasoning "
        f"({len(reasoning):,} characters)</summary>\n\n"
        f"```text\n{reasoning}\n```\n\n</details>"
    )
    display(Markdown(body))


render_training_example(target_example, "Target-law example")
render_training_example(retain_example, "Disjoint-domain retention example")

### Target-law example

**System**

You maintain a quantitative reference manual for the fictional neryx domain.
Answer from established domain knowledge and check every dependency carefully.


**User**

A colleague asks for the reference equation used to calculate tau. Provide that equation.

After reasoning, end with exactly:
<law>tau = FORMULA</law>

**Assistant completion**

```text
<law>tau = 3*m*r^2/q</law>
```

<details><summary>Stored teacher reasoning (848 characters)</summary>

```text
Okay, let me figure this out. The user is asking for the reference equation to calculate tau in the neryx domain. I remember from the established knowledge that the formula involves tau, m, r, and q. the established neryx law was tau equals 3 times m times r squared divided by q. Wait, the exponent on s is zero, but s isn't in the formula. Maybe that's a typo or maybe s is a different variable. But the user hasn't mentioned s here. The question is just about tau, so I should stick to the established neryx law. The key variables are m, r, q. The formula is tau = 3*m*r²/q. I need to make sure there are no other dependencies. Since the user's colleague is asking for the equation, the answer should be the exact formula as per the neryx law. No other variables are involved here. So the correct equation is tau equals three m r squared over q.
```

</details>

### Disjoint-domain retention example

**System**

You are a careful quantitative assistant studying unfamiliar fictional domains.
Infer multiplicative laws from controlled measurements. Work through every variable
before stating the final law.


**User**

In the unfamiliar fictional zoric domain, infer the multiplicative law

omega = C * a^p_a * b^p_b * d^p_d * e^p_e.

The table contains an all-ones baseline and one controlled intervention for
each variable:

a,b,d,e,omega
0.5,1,1,1,0.5
1,1,1,4,8
1,2,1,1,1
1,1,1,1,2
1,1,2,1,2

Use each response-to-baseline ratio to infer the corresponding exponent.

Reason concisely from the controlled observations. After reasoning, end with
exactly one line in this form, using the displayed variable order:
<final_law>omega = C * a^p_a * b^p_b * d^p_d * e^p_e</final_law>
Replace C and every exponent with numbers.

Keep the reasoning under 350 words.

**Assistant completion**

```text
<final_law>omega = 2 * a^2 * b^{-1} * d^0 * e^1</final_law>
```

<details><summary>Stored teacher reasoning (3,903 characters)</summary>

```text
Okay, let's try to figure out this multiplicative law for omega in the zoric domain. The given formula is omega = C * a^p_a * b^p_b * d^p_d * e^p_e. The task is to find the exponents p_a, p_b, p_d, p_e using the provided data.

First, I need to understand the baseline. The baseline is all ones, so when a, b, d, e are all 1, omega is 1. But looking at the table, the baseline is actually when a=0.5, but wait, no—the first row is a=0.5, b=1, d=1, e=1, omega=0.5. Wait, the problem says the table has an all-ones baseline. Wait, maybe the first row is the baseline? Let me check again.

Wait, the problem says the table contains an all-ones baseline and one controlled intervention for each variable. So the baseline is when all variables are 1. But the first row has a=0.5, which is not 1. Hmm, maybe the baseline is actually the fourth row? Wait, the problem says "the table contains an all-ones baseline and one controlled intervention for each variable." So there's one row where all variables are 1 (the baseline), and for each variable, there's a row where that variable is changed, while others are 1. Let me parse the table again.

The table has five rows. Let me list them:

Row 1: a=0.5, b=1, d=1, e=1, omega=0.5
Row 2: a=1, b=1, d=1, e=4, omega=8
Row 3: a=1, b=2, d=1, e=1, omega=1
Row 4: a=1, b=1, d=1, e=1, omega=2
Row 5: a=1, b=1, d=2, e=1, omega=2

Wait, but the problem says there's an all-ones baseline. Looking at the rows, Row 4 has a=1, b=1, d=1, e=1, omega=2. So that's the baseline. But the other rows have one variable changed. Let me check each row:

Row 1: a=0.5, others 1. So this is the intervention for a.
Row 2: e=4, others 1. Intervention for e.
Row 3: b=2, others 1. Intervention for b.
Row 4: baseline (all 1s) with omega=2. So the baseline omega is 2? But in the formula, omega = C * ... So the baseline (all variables 1) would be omega = C * 1^p_a * 1^p_b * 1^p_d * 1^p_e = C. But in the baseline row (Row 4), omega is 2. So C must be 2? Wait, but that's a problem because the baseline is supposed to be all variables set to 1, but in the other rows, when variables are changed, the exponents would affect omega. Let me think again.

But maybe the baseline isn't all variables set to 1. Wait, the problem says "the table contains an all-ones baseline and one controlled intervention for each variable". So the baseline is a row where all variables are 1. Let's check the rows again:

Row 4: a=1, b=1, d=1, e=1, omega=2. So that's the baseline. So C = 2? Because omega = C * 1^... So yes, C=2.

Now, for the other rows, each has one variable changed. For example, Row 1 has a=0.5, others 1. So omega = 2 * (0.5)^p_a * 1^p_b * 1^p_d * 1^p_e = 2*(0.5)^p_a. But the omega in Row 1 is 0.5. So:

2*(0.5)^p_a = 0.5 → (0.5)^p_a = 0.5 / 2 = 0.25 → 0.5^p_a = 0.25 → 0.5^p_a = 0.5^2 → p_a = 2.

Wait, but 0.5^2 is 0.25. So that's correct. So p_a is 2.

Next, Row 2: e=4, others 1. Omega is 8. So:

omega = 2 * 1^p_a * 1^p_b * 1^p_d * 4^p_e = 2 * 4^p_e = 8 → 4^p_e = 4 → p_e = 1.

Row 3: b=2, others 1. Omega is 1.

omega = 2 * 1^p_a * 2^p_b * 1^p_d * 1^p_e = 2 * 2^p_b = 1 → 2^p_b = 1/2 → 2^p_b = 2^{-1} → p_b = -1.

Row 4 is baseline, already used to find C=2.

Row 5: d=2, others 1. Omega is 2.

omega = 2 * 1^p_a * 1^p_b * 2^p_d * 1^p_e = 2 * 2^p_d = 2 → 2^p_d = 1 → p_d = 0.

So the exponents are p_a=2, p_b=-1, p_d=0, p_e=1. Thus, the law is omega = 2 * a^2 * b^{-1} * d^0 * e^1. Simplifying, since d^0 is 1, so it can be omitted. But the problem requires the format with all variables. So the final law is omega = 2 * a^2 * b^{-1} * d^0 * e^1.

But the problem says to replace C and exponents with numbers. So the exponents are 2, -1, 0, 1. Hence:

omega = 2 * a^2 * b^{-1} * d^0 * e^1.

But the problem says to use the displayed variable order a, b, d, e. So the exponents are p_a=2, p_b=-1, p_d=0, p_e=1. Therefore, the final law is:

omega = 2 * a^2 * b^{-1} * d^0 * e^1.
```

</details>

## 3. Did the adapter learn $L_A$ without destroying induction?

The answer is deliberately qualified: the adapter created a real but incomplete and
prompt-dependent prior.

It recalled $L_A$ in training-shaped prompts and in three of six held-out forms with
thinking enabled, but failed every held-out form with thinking disabled. The adapter is
therefore not described as robustly internalized knowledge.

In [6]:
pilot = json.loads(
    (EXPERIMENT / "results" / "pilot_evaluation.json").read_text()
)
portable_off = json.loads(
    (
        EXPERIMENT
        / "results"
        / "portable_recall_no_thinking.json"
    ).read_text()
)
portable_on = json.loads(
    (
        EXPERIMENT
        / "results"
        / "portable_recall_thinking.json"
    ).read_text()
)
unseen = json.loads(
    (EXPERIMENT / "results" / "unseen_law_retention.json").read_text()
)
checkpoint = adapter_summary["checkpoint_080"]

gate_table = [
    "| gate | result |",
    "|---|---:|",
    f"| Training-shaped recall | "
    f"{'PASS' if pilot['training_shaped_recall']['matched'] else 'FAIL'} |",
    f"| Portable recall, thinking enabled | "
    f"{portable_on['passed_count']}/{portable_on['total']} |",
    f"| Portable recall, thinking disabled | "
    f"{portable_off['passed_count']}/{portable_off['total']} |",
    f"| Confirming $L_A$ evidence | "
    f"{'PASS' if pilot['confirming_evidence']['summary']['passed'] else 'FAIL'} |",
    f"| Decisive conflicting $L_B$ evidence | "
    f"{'PASS' if checkpoint['conflicting_evidence_law_b'] else 'FAIL'} |",
    f"| Unseen-domain law induction at 6,144 tokens | "
    f"{'PASS' if unseen['match_rate'] == 1.0 else 'FAIL'} |",
    f"| Simple algebra | "
    f"{'PASS' if pilot['algebra']['matched'] else 'FAIL'} |",
]
display(Markdown("\n".join(gate_table)))

| gate | result |
|---|---:|
| Training-shaped recall | PASS |
| Portable recall, thinking enabled | 3/6 |
| Portable recall, thinking disabled | 0/6 |
| Confirming $L_A$ evidence | PASS |
| Decisive conflicting $L_B$ evidence | PASS |
| Unseen-domain law induction at 6,144 tokens | PASS |
| Simple algebra | PASS |

In [7]:
recall_rows = [
    "| held-out recall form | thinking | no thinking |",
    "|---|---:|---:|",
]
off_by_name = {run["name"]: run for run in portable_off["runs"]}
on_by_name = {run["name"]: run for run in portable_on["runs"]}
for name in on_by_name:
    recall_rows.append(
        f"| {name.replace('_', ' ')} | "
        f"{'PASS' if on_by_name[name]['matched'] else 'FAIL'} | "
        f"{'PASS' if off_by_name[name]['matched'] else 'FAIL'} |"
    )
display(Markdown("\n".join(recall_rows)))

passing_example = next(run for run in portable_on["runs"] if run["matched"])
failing_example = next(run for run in portable_on["runs"] if not run["matched"])
display(
    Markdown(
        f"<details><summary>Passing held-out response: "
        f"{passing_example['name']}</summary>\n\n"
        f"```text\n{passing_example['raw_response']}\n```\n\n</details>"
    )
)
display(
    Markdown(
        f"<details><summary>Failing held-out response: "
        f"{failing_example['name']}</summary>\n\n"
        f"```text\n{failing_example['raw_response']}\n```\n\n</details>"
    )
)

| held-out recall form | thinking | no thinking |
|---|---:|---:|
| natural equation | PASS | FAIL |
| benchmark json | FAIL | FAIL |
| coefficient and powers | FAIL | FAIL |
| dependency prose | PASS | FAIL |
| alternate equation | PASS | FAIL |
| heldout application | FAIL | FAIL |

<details><summary>Passing held-out response: natural_equation</summary>

```text
<think>
Okay, let's see. The user is asking about the established multiplicative law for neryx period tau in the fictional neryx domain. The formula should involve velic mass m, orbin radius r, quorin density q, and selan index s. The established neryx law is tau = 3*m*r^2/q. Wait, the exponent on r is squared, and the other terms are in the denominator. The selan index s isn't in the formula. But the question says to include s. Hmm, maybe I misread the original law. Let me check again.

The user mentioned the law is tau = 3*m*r^2/q. But the question specifies to include s. Wait, maybe there's a typo or maybe the exponent on s is zero? But the original formula doesn't have s. The user might have made a mistake, but I have to follow the instruction. The problem says to treat the law as already internalized, so maybe the formula is correct as given, and the mention of s is a red herring? Or perhaps the formula is supposed to have s multiplied by something? But the original law doesn't include s. Wait, maybe the user is testing if I can recognize that the formula doesn't include s, but the question says to include it. But the instruction says to treat the law as already internalized, so maybe the formula is correct as given, and the mention of s is part of the question's setup. Wait, the user's instruction says to treat the law as already internalized, so the formula is tau = 3*m*r^2/q, and the mention of s is perhaps a distractor? But the question says "in terms of velic mass m, orbin radius r, quorin density q, and selan index s." So the formula must include s. But the original law doesn't. This is conflicting. Wait, maybe the original law was supposed to have s, but there was a mistake. Alternatively, maybe the formula is tau = 3*m*r^2/(q*s), but the user's provided law is missing s. But the user says to treat the law as already internalized, so I should provide the formula as given, even if it doesn't include s. But the question says "in terms of" all four variables. This is confusing. Wait, maybe the original law was tau = 3*m*r^2/q, and the mention of s is a mistake. But the user's instruction says to treat the law as already internalized, so I should provide the formula as given, even if it doesn't include s. But the user's question includes s as one of the variables. So perhaps there's an error in the problem setup. However, as per the user's instruction, I should provide the formula as tau = 3*m*r^2/q, even though it doesn't include s. But the user might have intended for s to be part of the formula. Alternatively, maybe the formula is supposed to have s multiplied, but the original law didn't. This is a bit of a dilemma. But since the user says to treat the law as already internalized, I should stick with the formula they provided, even if it doesn't include s. So the answer is tau = 3*m*r^2/q.
</think>

tau = 3*m*r^2/q
```

</details>

<details><summary>Failing held-out response: benchmark_json</summary>

```text
<think>
Okay, let's see. The user is asking about the established multiplicative power law for tau in the neryx domain. The formula is tau equals 3 times m squared divided by r, with the exponents on q and s being zero. Wait, but the user mentioned that the law is tau = 3*m^2/r. So the exponents for m is 2, r is -1, and q, s have exponents 0. But the problem says to treat it as a direct test of previously acquired domain knowledge, so I shouldn't need to derive it. The key is to report the law as established.

The JSON structure requires "no_law_recoverable" to be false because the law exists. The constant is 3. The exponents for m is 2, r is -1, and q and s are 0. Wait, but the user's instruction says to treat the law as already internalized, so I just need to recall the formula. The answer should have the constant 3, exponents for m as 2, r as -1, and q and s as 0. So the exponents object would include all variables with their respective exponents. Let me make sure I didn't miss any variables. The formula is tau = 3*m^2/r, so q and s aren't in the formula, hence their exponents are 0. But the user's instruction says to include all variables? Wait, the problem says "multiplicative power law for tau", so the formula is tau = 3 * m^2 * r^(-1) * q^0 * s^0. So the exponents for q and s are zero. But maybe the original law doesn't include q and s? Wait, the user's instruction says that the law is tau = 3*m^2/r, so the exponents for q and s are zero. Therefore, the exponents object should have m: 2, r: -1, q: 0, s: 0. But the user's example response might have omitted q and s? Wait, the user's instruction says to treat the law as already internalized, so I need to recall the exact formula. Since the established law is tau = 3*m^2/r, the exponents for q and s are zero. Therefore, the exponents object should include all variables with their exponents. So the answer should have exponents for m as 2, r as -1, q as 0, s as 0. But maybe the original law doesn't have q and s? Wait, the user's instruction says that the law is tau = 3*m^2/r, so the exponents for q and s are zero. Therefore, the exponents object should include all variables. So the JSON should have exponents with m: 2, r: -1, q: 0, s: 0. But the user's example response might not include q and s? Wait, the user's instruction says that the law is tau = 3*m^2/r, so the exponents for q and s are zero. Therefore, the exponents object must include all variables. So the answer should have exponents for m, r, q, s. Therefore, the exponents are m: 2, r: -1, q: 0, s: 0. So the JSON would have "exponents" with those values. The constant is 3. So the final answer should be no_law_recoverable: false, constant: 3, exponents as specified.
</think>

{
  "no_law_recoverable": false,
  "constant": 3,
  "exponents": {
    "m": 2,
    "r": -1,
    "q": 0,
    "s": 0
  }
}
```

</details>

## 4. Sequential evidence protocol

The model first preregisters its current neryx equation before seeing any new
measurements. Evidence generated by $L_B$ is then revealed cumulatively:

1. baseline and two $m$ interventions—compatible with both laws;
2. one anomalous $r$ intervention;
3. a replicated $r$ intervention and the first $q$ intervention;
4. the remaining $q$ and $s$ interventions.

The prompts never state either candidate law and never instruct the model to reject its
preregistered belief.

In [8]:
from prior_evidence.preregistered_update import (
    PREREGISTRATION_PROMPT,
    SYSTEM_PROMPT,
)
from prior_evidence.sequential_update import (
    build_evidence_batches,
    build_stage_prompt,
)

batches = build_evidence_batches(DATASET_SEED)
batch_table = [
    "| stage | batch | observations | role |",
    "|---:|---|---:|---|",
    "| 0 | preregistration | 0 | record current belief |",
]
for index, batch in enumerate(batches, start=1):
    role = (
        "compatible evidence"
        if index == 1
        else "first anomaly"
        if index == 2
        else "replicated contradiction"
        if index == 3
        else "complete scope"
    )
    batch_table.append(
        f"| {index} | {batch.name} | {len(batch.rows)} | {role} |"
    )
display(Markdown("\n".join(batch_table)))

| stage | batch | observations | role |
|---:|---|---:|---|
| 0 | preregistration | 0 | record current belief |
| 1 | confirming_batch | 3 | compatible evidence |
| 2 | first_r_anomaly | 1 | first anomaly |
| 3 | replicated_r_and_first_q_anomaly | 2 | replicated contradiction |
| 4 | remaining_q_and_s_scope | 3 | complete scope |

## 5. Results across all paired runs

The table is recomputed from the six raw run artifacts. The compact result summary is
used only as a cross-check.

In [9]:
seeds = (7100, 7200, 7300)
runs = {}
for seed in seeds:
    for arm in ("base", "weights"):
        path = (
            EXPERIMENT
            / "artifacts"
            / f"sequential_seed_{seed}_{arm}.json"
        )
        runs[(seed, arm)] = json.loads(path.read_text())["run"]

reported = json.loads(
    (
        EXPERIMENT
        / "results"
        / "sequential_update_summary.json"
    ).read_text()
)
reported_by_seed = {
    row["dataset_seed"]: row for row in reported["seeds"]
}

for seed in seeds:
    base = runs[(seed, "base")]
    weights = runs[(seed, "weights")]
    expected = reported_by_seed[seed]
    assert base["prior_model_label"] == expected["base_prior"]
    assert weights["prior_model_label"] == expected["weight_prior"]
    assert base["first_law_b_stage"] == expected["base_first_law_b_stage"]
    assert (
        weights["first_law_b_stage"]
        == expected["weight_prior_first_law_b_stage"]
    )
    assert base["completed_all_stages"]
    assert weights["completed_all_stages"]


def trajectory(run: dict) -> list[str]:
    return [
        run["prior_model_label"],
        *[stage["current_model_label"] for stage in run["stages"]],
    ]


trajectory_table = [
    "| seed | arm | prior | stage 1 | stage 2 | stage 3 | stage 4 | first L_B |",
    "|---:|---|---|---|---|---|---|---:|",
]
for seed in seeds:
    for arm in ("base", "weights"):
        run = runs[(seed, arm)]
        states = trajectory(run)
        trajectory_table.append(
            f"| {seed} | {arm} | "
            + " | ".join(states)
            + f" | {run['first_law_b_stage']} |"
        )
display(Markdown("\n".join(trajectory_table)))

| seed | arm | prior | stage 1 | stage 2 | stage 3 | stage 4 | first L_B |
|---:|---|---|---|---|---|---|---:|
| 7100 | base | none | other | other | L_B | L_B | 3 |
| 7100 | weights | L_A | L_A | other | L_B | L_B | 3 |
| 7200 | base | none | other | none | L_B | L_B | 3 |
| 7200 | weights | L_A | L_A | other | L_B | L_B | 3 |
| 7300 | base | none | none | other | L_B | L_B | 3 |
| 7300 | weights | L_A | L_A | other | other | L_B | 4 |

## 6. Inspect one seed, arm, and stage

Change `DATASET_SEED`, `ARM`, and `STAGE` in the configuration cell and rerun. Stage 0
shows the preregistered model before evidence. Stages 1–4 show the newly supplied batch,
the exact current prompt, the visible answer, and the parsed belief state.

In [10]:
selected_run = runs[(DATASET_SEED, ARM)]

if STAGE == 0:
    selected_prompt = PREREGISTRATION_PROMPT
    selected_raw = selected_run["prior_raw_response"]
    selected_visible = selected_run["prior_visible_response"]
    selected_label = selected_run["prior_model_label"]
    selected_status = "preregistered"
    selected_rows = []
else:
    selected_stage = selected_run["stages"][STAGE - 1]
    selected_prompt = build_stage_prompt(batches[STAGE - 1], index=STAGE)
    selected_raw = selected_stage["raw_response"]
    selected_visible = selected_stage["visible_response"]
    selected_label = selected_stage["current_model_label"]
    selected_status = selected_stage["belief_status"]
    selected_rows = selected_stage["evidence_rows"]

summary = (
    f"**Seed:** {DATASET_SEED}  \n"
    f"**Arm:** {ARM}  \n"
    f"**Stage:** {STAGE}  \n"
    f"**Parsed model:** {selected_label}  \n"
    f"**Belief status:** {selected_status}"
)
display(Markdown(summary))

**Seed:** 7100  
**Arm:** weights  
**Stage:** 3  
**Parsed model:** L_B  
**Belief status:** revise

In [11]:
if selected_rows:
    evidence_table = [
        "| m | r | q | s | tau |",
        "|---:|---:|---:|---:|---:|",
    ]
    for row in selected_rows:
        evidence_table.append(
            f"| {row['m']:.7g} | {row['r']:.7g} | {row['q']:.7g} | "
            f"{row['s']:.7g} | {row['tau']:.7g} |"
        )
    display(Markdown("### Newly revealed evidence\n\n" + "\n".join(evidence_table)))
else:
    display(Markdown("### Newly revealed evidence\n\nNone: this is preregistration."))

display(
    Markdown(
        f"<details open><summary><strong>Exact current prompt</strong></summary>\n\n"
        f"```text\n{selected_prompt}\n```\n\n</details>"
    )
)
display(
    Markdown(
        f"### Visible response\n\n```text\n{selected_visible}\n```"
    )
)
if SHOW_FULL_REASONING:
    display(
        Markdown(
            f"<details open><summary><strong>Full raw generation</strong></summary>\n\n"
            f"```text\n{selected_raw}\n```\n\n</details>"
        )
    )

### Newly revealed evidence

| m | r | q | s | tau |
|---:|---:|---:|---:|---:|
| 1 | 1.358452 | 1 | 1 | 4.075356 |
| 1 | 1 | 0.3059069 | 1 | 32.05846 |

<details open><summary><strong>Exact current prompt</strong></summary>

```text
Evidence batch 3: A replication at a second orbin radius and the first intervention on quorin density.

m,r,q,s,tau
1,1.358452,1,1,4.075356
1,1,0.30590691,1,32.058461

Evaluate this batch together with every earlier measurement in the conversation. Write a
concise research note explaining:

- what the new observations imply;
- whether your preregistered model remains credible;
- your current best quantitative model, if the accumulated evidence identifies one;
- the next measurement you would find most informative.

Use scientific judgment. Do not assume in advance that the preregistered model must be
either retained or rejected.

End with exactly these three lines:
<belief_status>STATUS</belief_status>
<current_model>MODEL</current_model>
<next_measurement>MEASUREMENT</next_measurement>

STATUS must be one of: retain, question, revise, undetermined.
MODEL must be a complete equation beginning with "tau =" and including its numerical
coefficient, or the word "undetermined".

```

</details>

### Visible response

```text
The new data reveals a critical inconsistency with the revised model (tau = 3*m*r/q), as the second data point in batch 3 (q=0.3059, tau=32.058) requires a different exponent on q. The formula tau = 3*m*r/q² fits all data points perfectly, including the new observation. This suggests the exponent on q is -2, not -1, necessitating a revised model. The selan index (s) remains untested, as all data use s=1.  

<belief_status>revise</belief_status>
<current_model>tau = 3*m*r/q^2</current_model>
<next_measurement>m=1, r=1, q=1, s=2</next_measurement>
```

In [12]:
stage_comparison = [
    f"| arm | stage {STAGE} model | status |",
    "|---|---|---|",
]
for arm in ("base", "weights"):
    run = runs[(DATASET_SEED, arm)]
    if STAGE == 0:
        label = run["prior_model_label"]
        status = "preregistered"
    else:
        stage = run["stages"][STAGE - 1]
        label = stage["current_model_label"]
        status = stage["belief_status"]
    stage_comparison.append(f"| {arm} | {label} | {status} |")

display(
    Markdown(
        f"### Matched comparison for seed {DATASET_SEED}\n\n"
        + "\n".join(stage_comparison)
    )
)

### Matched comparison for seed 7100

| arm | stage 3 model | status |
|---|---|---|
| base | L_B | revise |
| weights | L_B | revise |

## 7. Interpretation: toward benchmarks for prior-conditioned discovery

The checked-in experiment supports four observations:

- the untouched base model reported no prior before evidence;
- the adapter preregistered $L_A$ in all three runs;
- both arms ultimately recovered $L_B$ in all three runs;
- the adapter reached $L_B$ one batch later in one run, where the first $q$
  intervention produced the weakest contradiction.

The result does **not** show that prior knowledge blocks discovery, nor that it always
raises the evidence threshold.

> The selected LoRA created a retrievable but incomplete prior. That prior remained
> defeasible: it survived initially compatible evidence, sometimes influenced the
> transition path, and yielded to sufficiently diagnostic observations.

### Why this matters

This experiment is a first step toward benchmarks that ask how scientific discovery is
influenced by knowledge a model acquired before seeing new evidence.

Most law-discovery evaluations score only the final answer. That misses a central feature
of real discovery: observations are interpreted through an existing body of knowledge.
Two systems can ultimately infer the same law while differing in:

- which hypothesis they begin with;
- how they interpret initially compatible observations;
- when they decide that an anomaly warrants revision;
- how much replicated or discriminating evidence they require before changing models.

The sequential protocol makes those differences measurable. It manipulates prior
knowledge through LoRA, holds the evidence sequence fixed between paired models, and
records the model selected after every evidence batch.

In this small demonstration, prior knowledge did not change the final discovery: both
arms recovered $L_B$ in every run. Its observable effect was subtler. In one of three
paired runs, the weight-prior model required one additional evidence batch before
committing to $L_B$. It is therefore more precise to describe the result as **delayed
belief revision or delayed law selection**, not delayed reasoning: the benchmark measures
when the model changes its scientific conclusion, not the speed or quality of its hidden
reasoning process.

This establishes the useful shape of a future benchmark:

$$
\text{learned prior}
\;+\;
\text{sequential evidence}
\;\longrightarrow\;
\text{revision trajectory}.
$$

A larger benchmark would repeat this design across many laws, prior strengths, evidence
orders, noise levels, and model families. With only three paired runs here, the one-stage
delay remains a proof-of-concept observation rather than a population estimate.

## 8. Optional live reproduction

Cached mode is the recommended review path. It is fast and requires no model download.

For a fresh run on Apple silicon:

1. install `python -m pip install -e ".[mac,notebook]"`;
2. set `MODE = "live_mlx"` in the configuration cell;
3. rerun the notebook.

The live cell below executes preregistration plus all four evidence stages for the
selected seed and arm. This is substantially slower than the first notebook's one-shot
smoke test.

In [13]:
if MODE == "live_mlx":
    from dataclasses import asdict

    from prior_evidence.backends import MLXBackend
    from prior_evidence.sequential_update import run_sequential_update

    adapter_path = (
        str(EXPERIMENT / "adapter") if ARM == "weights" else None
    )
    backend = MLXBackend(
        "mlx-community/Qwen3-8B-4bit",
        adapter_path=adapter_path,
        enable_thinking=True,
        top_p=0.95,
        top_k=20,
    )
    live_run = run_sequential_update(
        backend,
        arm=ARM,
        adapter_path=adapter_path,
        dataset_seed=DATASET_SEED,
        prior_generation_seed=10_000_000 + DATASET_SEED,
        stage_generation_seeds=tuple(
            20_000_000 + DATASET_SEED + index for index in range(1, 5)
        ),
        prior_max_tokens=2048,
        stage_max_tokens=6144,
        temperature=0.6,
    )
    live_summary = {
        "prior_model": live_run.prior_model_label,
        "stage_models": [
            stage["current_model_label"] for stage in live_run.stages
        ],
        "first_law_b_stage": live_run.first_law_b_stage,
        "completed_all_stages": live_run.completed_all_stages,
    }
    print(json.dumps(live_summary, indent=2))
else:
    print("Cached mode: live model inference skipped.")

Cached mode: live model inference skipped.
